# Exploration des différentes sessions

In [1]:
import pandas as pd
from tqdm import tqdm
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

In [2]:
boc = BrainObservatoryCache(manifest_file="./data/manifest.json")
data = boc.get_ophys_experiments()
sessions = pd.DataFrame(data)
print(f"{sessions.shape[0]} sessions, {sessions.shape[1]} colonnes")
sessions.head(5)

2534 sessions, 11 colonnes


,id,imaging_depth,targeted_structure,cre_line,reporter_line,acquisition_age_days,experiment_container_id,session_type,donor_name,specimen_name,fail_eye_tracking
0,991852000,171,VISp,Slc17a7-IRES2-Cre,Ai93(TITL-GCaMP6f),171,1018028049,OPHYS_6_images_B,479839,Slc17a7-IRES2-Cre;Camk2a-tTA;Ai93-479839,True
1,649409874,175,VISpm,Vip-IRES-Cre,Ai148(TIT2L-GC6f-ICL-tTA2),100,646959440,three_session_A,350249,Vip-IRES-Cre;Ai148-350249,False
2,663488086,175,VISl,Slc17a7-IRES2-Cre,Ai93(TITL-GCaMP6f),124,662358769,three_session_B,361636,Slc17a7-IRES2-Cre;Camk2a-tTA;Ai93-361636,False
3,958741219,150,VISp,Sst-IRES-Cre,Ai148(TIT2L-GC6f-ICL-tTA2),216,1018028342,OPHYS_3_images_A,457841,Sst-IRES-Cre;Ai148-457841,True
4,939319851,175,VISp,Vip-IRES-Cre,Ai148(TIT2L-GC6f-ICL-tTA2),141,928325203,OPHYS_5_images_A_passive,467951,Vip-IRES-Cre;Ai148-467951,True


🧠 Session / Experiment Identifiers  
* id  
  * Unique identifier for that specific imaging session (one experiment).  
* experiment_container_id  
  * Groups together multiple sessions from the same experiment series (e.g., the same mouse imaged across different session types).  

🔬 Brain & Imaging Details  
* imaging_depth  
  * Depth (in microns) below the brain surface where imaging was performed.  
  * Gives you an idea of cortical layer (e.g., ~175 µm = layer 2/3, deeper = layer 4/5).  
* targeted_structure  
  * Brain region that was imaged (e.g., VISp = primary visual cortex).  

🧬 Genetic / Biological Info  
* cre_line  
  * The genetically modified mouse line expressing Cre recombinase.  
  * This determines which neuron types are labeled (e.g., excitatory vs inhibitory subtypes).  
* reporter_line  
  * The fluorescent indicator line used (typically expressing something like GCaMP for calcium imaging).  

🐭 Subject Information  
* donor_name  
  * Identifier for the mouse (the biological donor).  
* specimen_name  
  * Identifier for the specific specimen/sample—often closely related to donor but used in internal tracking.  
* acquisition_age_days  
  * Age of the mouse (in days) at the time of the imaging session.  

🎬 Experimental Context  
* session_type  
  * The type of stimulus/experiment used during imaging.  
  * Examples include:  
    * drifting gratings  
    * natural scenes  
    * spontaneous activity  
  * This is crucial for interpreting neural responses.  

⚠️ Quality / Data Flags  
* fail_eye_tracking  
  * Boolean flag (True / False) indicating whether eye tracking data failed or is unreliable for that session.  
  * Important if you’re analyzing behavior or visual attention.  

--- 

Source : ChatGPT

## Types de sessions

In [3]:
sessions.session_type.value_counts(dropna=False)

three_session_A                    456
three_session_B                    456
three_session_C2                   380
OPHYS_3_images_A                   182
OPHYS_4_images_B                   181
OPHYS_1_images_A                   178
OPHYS_2_images_A_passive           146
OPHYS_6_images_B                   141
OPHYS_5_images_B_passive           132
three_session_C                     76
OPHYS_6_images_A                    41
OPHYS_4_images_A                    38
OPHYS_3_images_B                    34
OPHYS_2_images_B_passive            33
OPHYS_1_images_B                    31
OPHYS_5_images_A_passive            28
OPHYS_7_receptive_field_mapping      1
Name: session_type, dtype: int64

🔬 Common ophys session types (Visual Coding dataset)  
1. Three-session pipeline (classic Visual Coding)  

These are often labeled something like A, B, C sessions:  

🅰️ Session A — Drifting gratings  
Stimulus: moving sinusoidal gratings at different orientations, spatial/temporal frequencies  
Purpose:  
Measure orientation selectivity  
Measure direction selectivity  
What it tells you:  
→ Basic tuning properties of neurons in visual cortex  

🅱️ Session B — Static gratings + natural images  
Stimuli:  
Static gratings (varying orientation/spatial freq)  
Natural images (flashed)  
Purpose:  
Study spatial feature tuning  
Compare responses to artificial vs natural stimuli  
What it tells you:  
→ Feature selectivity and population coding  

🅲 Session C — Natural movies  
Stimulus: continuous natural scene movies  
Purpose:  
Examine temporal dynamics  
Measure reliability/reproducibility  
What it tells you:  
→ How neurons encode dynamic, real-world input  


2. Session naming variants  

Depending on dataset version, you might see labels like:  

three_session_A  
three_session_B  
three_session_C  

These map directly to the A/B/C descriptions above.  


3. Functional interpretation across sessions  
Session	Stimulus type	Main question answered  
A	Drifting gratings	What orientation/direction does this neuron prefer?  
B	Static + images	What spatial features/images drive responses?  
C	Natural movies	How does the neuron respond over time to real scenes?  
🔁 Why multiple sessions exist  

The same neurons are often recorded across sessions, allowing you to:  

Compare tuning vs natural responses  
Study stability of neural coding  
Link simple features → complex representations  


⚠️ Important nuance  

Not all datasets use exactly the same structure:  

🧬 Visual Behavior / Neuropixels variants  

In newer datasets from the Allen Institute, session types may instead encode:  

Behavioral context (e.g., active task vs passive viewing)  
Familiar vs novel images  
Task stages (training vs imaging)  

So session names can look like:  

OPHYS_1_images_A  
OPHYS_2_images_B  
etc.  

In those cases:  

Numbers often indicate training stage or exposure  
Letters indicate stimulus set identity  

---

Source : ChatGPT

Pour nos expériences, on se concentre sur les sessions de type B. On peut utiliser three_session_B. 

### Régions du cerveau ciblées selon le type de session

In [4]:
sessions["COUNT"] = 1

In [5]:
pd.pivot_table(sessions,values="COUNT",index="session_type",columns="targeted_structure",aggfunc="sum",fill_value=0)

targeted_structure,VISal,VISam,VISl,VISp,VISpm,VISrl
session_type,,,,,,
OPHYS_1_images_A,0,0,63,115,0,0
OPHYS_1_images_B,0,0,0,31,0,0
OPHYS_2_images_A_passive,0,0,58,88,0,0
OPHYS_2_images_B_passive,0,0,2,31,0,0
OPHYS_3_images_A,0,0,73,109,0,0
OPHYS_3_images_B,0,0,0,34,0,0
OPHYS_4_images_A,0,0,0,38,0,0
OPHYS_4_images_B,0,0,70,111,0,0
OPHYS_5_images_A_passive,0,0,0,28,0,0


Les expériences OPHYS B semblent cibler la région 'VISp' plus particulièrement. C'est peut-être la plus intéressante. 

## Sélection de sessions

### Nombre de sessions d'intérêt selon imaging_depth

In [6]:
sessions.query("session_type=='three_session_B' and targeted_structure=='VISp'").imaging_depth.value_counts(dropna=False)

275    49
375    32
175    29
350    15
550     6
400     2
335     2
205     2
225     1
285     1
200     1
265     1
195     1
390     1
185     1
Name: imaging_depth, dtype: int64

Les expériences à une profondeur de 275 sont les plus fréquentes (mais est-ce que ce sont les plus intéressantes...?)

### Nombre de sessions d'intérêt par cre_line

In [7]:
sessions.query("session_type=='three_session_B' and targeted_structure=='VISp' and imaging_depth==275").cre_line.value_counts(dropna=False)

Vip-IRES-Cre         8
Rorb-IRES2-Cre       8
Cux2-CreERT2         8
Sst-IRES-Cre         7
Pvalb-IRES-Cre       7
Slc17a7-IRES2-Cre    6
Emx1-IRES-Cre        3
Scnn1a-Tg3-Cre       2
Name: cre_line, dtype: int64

### Nombre de sessions d'intérêt par âge

In [8]:
sessions2 = sessions.query("session_type=='three_session_B' and targeted_structure=='VISp' and imaging_depth==275").copy()
sessions2["AGE_BIN"] = pd.cut(sessions2.acquisition_age_days,bins=10,right=False,include_lowest=True)

In [9]:
pd.pivot_table(sessions2,values="COUNT",index="cre_line",columns="AGE_BIN",aggfunc="sum",fill_value=0)

AGE_BIN,"[77.0, 84.8)","[84.8, 92.6)","[92.6, 100.4)","[100.4, 108.2)","[108.2, 116.0)","[116.0, 123.8)","[123.8, 131.6)","[131.6, 139.4)","[139.4, 147.2)","[147.2, 155.078)"
cre_line,,,,,,,,,,
Cux2-CreERT2,0,2,1,3,1,1,0,0,0,0
Emx1-IRES-Cre,0,0,1,0,0,0,1,0,0,1
Pvalb-IRES-Cre,1,0,2,3,1,0,0,0,0,0
Rorb-IRES2-Cre,1,0,0,3,4,0,0,0,0,0
Scnn1a-Tg3-Cre,1,0,0,0,0,1,0,0,0,0
Slc17a7-IRES2-Cre,0,0,1,2,1,1,0,1,0,0
Sst-IRES-Cre,0,1,1,0,2,3,0,0,0,0
Vip-IRES-Cre,0,0,2,1,2,1,2,0,0,0


L'âge n'est peut-être pas aussi important que le profil génétique.

## Une session

In [10]:
sessions.query("session_type=='three_session_B' and targeted_structure=='VISp' and imaging_depth==275 and cre_line=='Rorb-IRES2-Cre' and acquisition_age_days>100")

,id,imaging_depth,targeted_structure,cre_line,reporter_line,acquisition_age_days,experiment_container_id,session_type,donor_name,specimen_name,fail_eye_tracking,COUNT
361,528480613,275,VISp,Rorb-IRES2-Cre,Ai93(TITL-GCaMP6f),115,526481129,three_session_B,244884,Rorb-IRES2-Cre;Camk2a-tTA;Ai93-244884,False,1
539,642651898,275,VISp,Rorb-IRES2-Cre,Ai93(TITL-GCaMP6f),115,642651896,three_session_B,339312,Rorb-IRES2-Cre;Camk2a-tTA;Ai93-339312,True,1
871,587071892,275,VISp,Rorb-IRES2-Cre,Ai93(TITL-GCaMP6f),102,586351979,three_session_B,304756,Rorb-IRES2-Cre;Camk2a-tTA;Ai93-304756,True,1
1083,590565013,275,VISp,Rorb-IRES2-Cre,Ai93(TITL-GCaMP6f),113,590168381,three_session_B,305467,Rorb-IRES2-Cre;Camk2a-tTA;Ai93-305467,True,1
1334,509962140,275,VISp,Rorb-IRES2-Cre,Ai93(TITL-GCaMP6f),103,511510675,three_session_B,228786,Rorb-IRES2-Cre;Camk2a-tTA;Ai93-228786,False,1
1417,501567237,275,VISp,Rorb-IRES2-Cre,Ai93(TITL-GCaMP6f),103,511510989,three_session_B,222431,Rorb-IRES2-Cre;Camk2a-tTA;Ai93-222431,True,1
1839,531124922,275,VISp,Rorb-IRES2-Cre,Ai93(TITL-GCaMP6f),109,530739574,three_session_B,249122,Rorb-IRES2-Cre;Camk2a-tTA;Ai93-249122,False,1


In [ ]:
for session_id in tqdm(sessions.query("session_type=='three_session_B' and targeted_structure=='VISp' and imaging_depth==275 and cre_line=='Rorb-IRES2-Cre' and acquisition_age_days>100").id):
    boc.get_ophys_experiment_data(session_id)